In [24]:
import pandas as pd

df = pd.read_csv("eight_mimic.csv")
df = df.drop(columns={"Unnamed: 0", "Unnamed: 0.1"})
df.columns

Index(['dicom_id', 'subject_id', 'study_id',
       'PerformedProcedureStepDescription', 'ViewPosition', 'Rows', 'Columns',
       'StudyDate', 'StudyTime', 'ProcedureCodeSequence_CodeMeaning',
       'ViewCodeSequence_CodeMeaning',
       'PatientOrientationCodeSequence_CodeMeaning', 'p', 'FileName', 'Split',
       'report_path', 'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema',
       'Enlarged Cardiomediastinum', 'Fracture', 'Lung Lesion', 'Lung Opacity',
       'No Finding', 'Pleural Effusion', 'Pleural Other', 'Pneumonia',
       'Pneumothorax', 'Support Devices', 'impression', 'Finding Label',
       'Finding Labels'],
      dtype='object')

In [25]:
len(df)

119534

In [26]:
len(df["subject_id"].unique())

48496

In [27]:
import numpy as np

mapping = {subid.astype(int).item(): i.item() for subid, i in zip(df["subject_id"].unique(), np.arange(len(df["subject_id"].unique())))}
len(mapping)

48496

In [28]:
df["id"] = df["subject_id"].map(mapping)
df[["id", "subject_id"]].tail(20)

,id,subject_id
119514,48486,10776100.0
119515,48487,10773964.0
119516,48488,10771901.0
119517,48489,10770325.0
119518,48489,10770325.0
119519,48490,10769360.0
119520,48491,10768869.0
119521,48492,10768638.0
119522,48493,10765994.0
119523,48493,10765994.0


In [29]:
df["Split"].value_counts()

Split
TRAIN    91843
TEST     16125
VAL      11566
Name: count, dtype: int64

In [30]:
df = df.rename(columns={"FileName": "path"})
df = df[['id', 'path', 'Split', 'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema',
       'Enlarged Cardiomediastinum', 'Fracture', 'Lung Lesion', 'Lung Opacity',
       'No Finding', 'Pleural Effusion', 'Pleural Other', 'Pneumonia',
       'Pneumothorax', 'Support Devices']]

In [ ]:

df.to_csv("../mimic.csv")

# MIMIC -- But images are annoyingly large. reduce to 512 

In [ ]:
from PIL import Image 
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm
import os
from concurrent.futures import ThreadPoolExecutor
import multiprocessing

# Function to process single image
def process_image(args):
    input_path, output_path, size = args
    try:
        img = Image.open(input_path).convert('RGB')
        
        # Calculate dimensions for center crop
        width, height = img.size
        if width > height:
            left = (width - height) // 2
            top = 0
            right = left + height
            bottom = height
        else:
            top = (height - width) // 2
            left = 0
            bottom = top + width
            right = width

        # Center crop and resize
        img_cropped = img.crop((left, top, right, bottom))
        img_resized = img_cropped.resize((size, size), Image.Resampling.LANCZOS)
        
        # Save as PNG
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        output_path = output_path.rsplit('.', 1)[0] + '.png'  # Change extension to png
        img_resized.save(output_path, 'PNG')
        return None
    except Exception as e:
        return f"Error processing {input_path}: {str(e)}"

# Load CSV file
df = pd.read_csv("/vol/ideadata/ed52egek/pycharm/syneverything/datasets/mimic.csv")
base_input_dir = "/vol/ideadata/ed52egek/pycharm/syneverything/datasets/data"
base_output_dir = "/vol/ideadata/ed52egek/pycharm/syneverything/datasets/data/processed_files/"

# Prepare arguments for parallel processing
process_args = []
for _, row in df.iterrows():
    input_path = os.path.join(base_input_dir, row['path'])
    output_path = os.path.join(base_output_dir, row['path'])
    process_args.append((input_path, output_path, 512))

# Use number of CPU cores for processing
num_workers = 4#multiprocessing.cpu_count()

# Process images in parallel with progress bar
with ThreadPoolExecutor(max_workers=num_workers) as executor:
    results = list(tqdm(
        executor.map(process_image, process_args),
        total=len(process_args),
        desc=f"Processing images with {num_workers} workers"
    ))

# Print any errors that occurred
errors = [r for r in results if r is not None]
if errors:
    print("\nErrors encountered:")
    for error in errors:
        print(error)

print("Processing complete!")


,id,path,Split,Atelectasis,Cardiomegaly,Consolidation,Edema,Enlarged Cardiomediastinum,Fracture,Lung Lesion,Lung Opacity,No Finding,Pleural Effusion,Pleural Other,Pneumonia,Pneumothorax,Support Devices
0,0,files/p19/p19999987/s58971208/1a1fe7e3-cbac5d9...,TRAIN,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0,files/p19/p19999987/s58621812/7ba273af-3d290f8...,TRAIN,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,1,files/p19/p19999733/s57132437/3fcd0406-9b11160...,TRAIN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
3,1,files/p19/p19999733/s57132437/428e2c18-5721d8f...,TRAIN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4,2,files/p19/p19999376/s57540554/53e9b6d0-5d5317f...,TRAIN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119529,48495,files/p10/p10760670/s59575239/926ca783-1abb560...,TEST,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
119530,48495,files/p10/p10760670/s56785501/068144bb-0b4fa96...,TEST,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
119531,48495,files/p10/p10760670/s54827584/6c3436b6-65eeb5b...,TEST,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0
119532,48495,files/p10/p10760670/s53468449/aa1de98c-5b2943a...,TEST,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0


In [ ]:
# Create new DataFrame with updated paths
df_512 = df.copy()

# Drop Unnamed: 0 column if it exists
if 'Unnamed: 0' in df_512.columns:
    df_512 = df_512.drop('Unnamed: 0', axis=1)

# Update paths to point to processed PNG files
df_512['path'] = df_512['path'].apply(lambda x: os.path.join('processed_files', x.rsplit('.', 1)[0] + '.png'))

# Save to new CSV file
output_csv_path = "/vol/ideadata/ed52egek/pycharm/syneverything/datasets/mimic512.csv"
df_512.to_csv(output_csv_path, index=False)

print(f"Created new CSV at {output_csv_path}")

